<a href="https://colab.research.google.com/github/rpizarrog/Libro-Aprendizaje-Automatico.-Casos-de-Estudio-con-R-y-Python/blob/main/Python%20NoteBooks/Regresi%C3%B3n_lineal_m%C3%B9%C3%BAtiple_con_datos_de_firjol_pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Contexto de los datos

El conjunto de datos corresponde a variables agronómicas medidas durante un ciclo de cultivo de frijol, cuyo objetivo es analizar los factores que influyen en la calidad del producto final, son alrededor de *1200* registros, estas son las variables independientes:

* *temperatura* es el promedio de la temperatura ambiental en un ciclo periodo de tiempo       
* *humedad* es el promedio de humedad de suelo en un ciclo de cultivo o periodo de tiempo   
* *precipitacion* es la cantidad de agua precipitada medida en un periodo de tiempo
* *nitrogeno* cantidad promedio de nitrógeno vertido como fertilizante promedio en un periodo de tiempo
* *fosforo* cantidad de promedio de fósfoto vertido como fertilizante en un periodo de tiempo
* *materia_organica* porcentaje de materia orgánica presente en el suelo, es como fertilziante.
* *horas_sol* cantidad de solo promedo medido en horas/dia  
* *tipo_suelo_arenoso* FALSO o VERDADERO su el tipo de suelo de los datos recabados es arenoso
* *tipo_suelo_franco* FALSO o VERDADERO su el tipo de suelo de los datos recabados es suelo franco o arcilloso.

La variable dependiente es *calidad_frijol* medido con valores aproximados entre 170 y 300, siendo valores altos de buena calidad y lo contrario con valores bajos.

El documento *notebook* se puede encontrar en el servicio *google collab* https://colab.research.google.com/drive/1AagRsq5NWuwFmLhwW250Bvaq_1RsgGJA?usp=sharing

Los datos para su descarga se encuentran en el servico *github.com* https://raw.githubusercontent.com/rpizarrog/Libro-Aprendizaje-Automatico.-Casos-de-Estudio-con-R-y-Python/refs/heads/main/datos/datos_frijol.csv.

Las funciones se pueden encontrar en **pendiente**



# Objetivo

Implementar y evaluar un modelo de regresión lineal múltiple que permita predecir calidad de frijol mediante programación en *Python*.

Los datos deberán estar particionados en *70%* para datos de entrenamiento y *30%* para datos de validación

El modelo será óptimo si satisface los postulados de la regresión y si el valor de *r square adjustado* está por encima del *85%*.

# Descripción

Se sigue la metodología sugerida para todos los casos de estudio.

## Cargar librerías



In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import PolynomialFeatures

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LassoCV
from sklearn.linear_model import RidgeCV

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from scipy.stats import shapiro
from scipy.stats import kstest

from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import linear_reset

import statsmodels.api as sm

## Cargar funciones


In [10]:
# Funciones para implementar y evaluar modelos de regresión múltiple en Python
# Rubén Pizarro Gurrola
# Mayo 2026

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LassoCV
from sklearn.linear_model import RidgeCV
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from scipy.stats import shapiro
from scipy.stats import kstest
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import linear_reset
import statsmodels.api as sm

#========================================================
# CARGAR DATOS
#========================================================

def f_cargar_datos(ruta_archivo):

    datos = pd.read_csv(ruta_archivo)

    return datos

#========================================================
# VISUALIZAR HEAD Y TAIL
#========================================================

def f_visualizar_head_tail_reducido(
        datos,
        n = 6
):

    #----------------------------------------------------
    # Total columnas
    #----------------------------------------------------

    total_columnas = datos.shape[1]

    #----------------------------------------------------
    # Primeras 4 columnas
    #----------------------------------------------------

    idx_prim = list(
        range(
            min(4, total_columnas)
        )
    )

    #----------------------------------------------------
    # Últimas 4 columnas
    #----------------------------------------------------

    idx_ult = list(
        range(
            max(total_columnas - 4, 0),
            total_columnas
        )
    )

    #----------------------------------------------------
    # Evitar duplicados
    #----------------------------------------------------

    idx_ult = [
        i for i in idx_ult
        if i not in idx_prim
    ]

    #----------------------------------------------------
    # Subconjuntos
    #----------------------------------------------------

    datos_prim = datos.iloc[:, idx_prim]

    datos_ult = datos.iloc[:, idx_ult]

    #----------------------------------------------------
    # HEAD
    #----------------------------------------------------

    head_prim = (
        datos_prim
        .head(n)
        .astype(str)
        .reset_index(drop = True)
    )

    head_ult = (
        datos_ult
        .head(n)
        .astype(str)
        .reset_index(drop = True)
    )

    #----------------------------------------------------
    # TAIL
    #----------------------------------------------------

    tail_prim = (
        datos_prim
        .tail(n)
        .astype(str)
        .reset_index(drop = True)
    )

    tail_ult = (
        datos_ult
        .tail(n)
        .astype(str)
        .reset_index(drop = True)
    )

    #----------------------------------------------------
    # Separadores
    #----------------------------------------------------

    sep_head = pd.DataFrame({
        "...": ["..."] * n
    })

    sep_tail = pd.DataFrame({
        "...": ["..."] * n
    })

    #----------------------------------------------------
    # Combinar HEAD
    #----------------------------------------------------

    head_comb = pd.concat(

        [
            head_prim,
            sep_head,
            head_ult
        ],

        axis = 1
    )

    #----------------------------------------------------
    # Combinar TAIL
    #----------------------------------------------------

    tail_comb = pd.concat(

        [
            tail_prim,
            sep_tail,
            tail_ult
        ],

        axis = 1
    )

    #----------------------------------------------------
    # Fila separadora
    #----------------------------------------------------

    fila_sep = pd.DataFrame(

        [["..."] * head_comb.shape[1]],

        columns = head_comb.columns
    )

    #----------------------------------------------------
    # Tabla final
    #----------------------------------------------------

    tabla = pd.concat(

        [
            head_comb,
            fila_sep,
            tail_comb
        ],

        ignore_index = True
    )

    return tabla

#========================================================
# DESCRIBIR DATOS
#========================================================

def f_describir_datos(datos):

    describe = datos.describe(include = 'all')

    structure = datos.dtypes

    return {
        "describe": describe,
        "structure": structure
    }

#========================================================
# PARTICIONAR DATOS
#========================================================

def f_particionar_datos(datos,
                         proporcion_entrenamiento = 0.7):

    datos_entrenamiento,
    datos_validacion = train_test_split(

        datos,

        train_size = proporcion_entrenamiento,

        random_state = 2026
    )

    return {
        "datos_entrenamiento": datos_entrenamiento,
        "datos_validacion": datos_validacion
    }

#========================================================
# CONVERTIR FACTOR
#========================================================

def f_convertir_factor(datos):

    datos_mod = datos.copy()

    for col in datos_mod.columns:

        if datos_mod[col].dtype == 'object':

            datos_mod[col] = datos_mod[col].astype('category')

        if datos_mod[col].dtype == 'bool':

            datos_mod[col] = datos_mod[col].astype(int)

    return datos_mod

#========================================================
# REDONDEAR VARIABLES NUMÉRICAS
#========================================================

def f_redondear_numericas(datos,
                          decimales = 2):

    datos_out = datos.copy()

    columnas_num = datos_out.select_dtypes(include = np.number).columns

    datos_out[columnas_num] = datos_out[columnas_num].round(decimales)

    return datos_out

#========================================================
# MODELO REGRESIÓN LINEAL MÚLTIPLE
#========================================================

def f_construir_modelo_RLM(datos,
                           variable_dependiente,
                           ver_resumen = True):

    X = datos.drop(columns = [variable_dependiente])

    y = datos[variable_dependiente]

    X = pd.get_dummies(X,
                       drop_first = True)

    modelo = LinearRegression()

    modelo.fit(X, y)

    if ver_resumen:

        X_sm = sm.add_constant(X)

        modelo_sm = sm.OLS(y, X_sm).fit()

        print(modelo_sm.summary())

    return modelo

#========================================================
# MODELO POLINOMIAL MÚLTIPLE
#========================================================

def f_multiple_polinomial(datos,
                          variable_dependiente,
                          orden = 2,
                          ver_resumen = True):

    X = datos.drop(columns = [variable_dependiente])

    y = datos[variable_dependiente]

    X = pd.get_dummies(X,
                       drop_first = True)

    poly = PolynomialFeatures(
        degree = orden,
        include_bias = False
    )

    X_poly = poly.fit_transform(X)

    modelo = LinearRegression()

    modelo.fit(X_poly, y)

    if ver_resumen:

        print("\n============================")
        print(f"Modelo Polinomial Orden {orden}")
        print("============================")

        print("Número de términos:", X_poly.shape[1])

    return {
        "modelo": modelo,
        "poly": poly
    }

#========================================================
# ESTANDARIZAR Y ESCALAR
#========================================================

def f_estandarizar_escalar(datos,
                           decimales = 4):

    datos_est = datos.copy()
    datos_esc = datos.copy()

    columnas_num = datos.select_dtypes(include = np.number).columns

    scaler_est = StandardScaler()

    scaler_minmax = MinMaxScaler()

    datos_est[columnas_num] = np.round(
        scaler_est.fit_transform(datos[columnas_num]),
        decimales
    )

    datos_esc[columnas_num] = np.round(
        scaler_minmax.fit_transform(datos[columnas_num]),
        decimales
    )

    return {
        "datos_estandarizados": datos_est,
        "datos_escalados": datos_esc
    }

#========================================================
# MODELO LASSO
#========================================================

def f_construir_modelo_lasso(datos,
                             variable_dependiente,
                             ver_resumen = True):

    X = datos.drop(columns = [variable_dependiente])

    y = datos[variable_dependiente]

    X = pd.get_dummies(X,
                       drop_first = True)

    modelo = LassoCV(
        cv = 10,
        random_state = 2026
    )

    modelo.fit(X, y)

    if ver_resumen:

        print("\n============================")
        print("Modelo LASSO")
        print("============================")

        print("Lambda óptimo:", modelo.alpha_)

        print("Coeficientes:")
        print(modelo.coef_)

    return modelo

#========================================================
# MODELO RIDGE
#========================================================

def f_construir_modelo_ridge(datos,
                             variable_dependiente,
                             ver_resumen = True):

    X = datos.drop(columns = [variable_dependiente])

    y = datos[variable_dependiente]

    X = pd.get_dummies(X,
                       drop_first = True)

    alphas = np.logspace(-4, 4, 100)

    modelo = RidgeCV(
        alphas = alphas,
        cv = 10
    )

    modelo.fit(X, y)

    if ver_resumen:

        print("\n============================")
        print("Modelo RIDGE")
        print("============================")

        print("Lambda óptimo:", modelo.alpha_)

        print("Coeficientes:")
        print(modelo.coef_)

    return modelo

#========================================================
# MULTICOLINEALIDAD VIF
#========================================================

def f_multicolinealidad(datos,
                        variable_dependiente):

    X = datos.drop(columns = [variable_dependiente])

    X = pd.get_dummies(X,
                       drop_first = True)

    X = sm.add_constant(X)

    vif = pd.DataFrame()

    vif["Variable"] = X.columns

    vif["VIF"] = [
        variance_inflation_factor(X.values, i)
        for i in range(X.shape[1])
    ]

    return vif

#========================================================
# LINEALIDAD
#========================================================

def f_linealidad(modelo,
                 X,
                 y):

    pred = modelo.predict(X)

    residuos = y - pred

    plt.figure(figsize = (6,4))

    plt.scatter(pred,
                residuos)

    plt.axhline(0,
                linestyle = '--')

    plt.title('Residuos vs Ajustados')

    plt.xlabel('Valores ajustados')

    plt.ylabel('Residuos')

    plt.show()

#========================================================
# TEST RAMSEY RESET
#========================================================

def f_linealidad_test(X,
                      y):

    X_sm = sm.add_constant(X)

    modelo = sm.OLS(y, X_sm).fit()

    resultado = linear_reset(
        modelo,
        power = 2,
        use_f = True
    )

    print(resultado)

    return resultado

#========================================================
# HOMOCEDASTICIDAD
#========================================================

def f_homocedasticidad(X,
                       y):

    X_sm = sm.add_constant(X)

    modelo = sm.OLS(y, X_sm).fit()

    residuos = modelo.resid

    bp = het_breuschpagan(
        residuos,
        X_sm
    )

    resultado = {
        "LM Statistic": bp[0],
        "LM p-value": bp[1],
        "F Statistic": bp[2],
        "F p-value": bp[3]
    }

    print(resultado)

    return resultado

#========================================================
# NORMALIDAD
#========================================================

def f_normalidad(residuos):

    residuos = np.array(residuos)

    shapiro_test = shapiro(residuos)

    ks_test = kstest(
        residuos,
        'norm',
        args = (
            np.mean(residuos),
            np.std(residuos)
        )
    )

    resultado = pd.DataFrame({
        "Prueba": [
            "Shapiro-Wilk",
            "Kolmogorov-Smirnov"
        ],
        "p_value": [
            shapiro_test.pvalue,
            ks_test.pvalue
        ]
    })

    return resultado

#========================================================
# INDEPENDENCIA
#========================================================

def f_independencia(residuos):

    dw = durbin_watson(residuos)

    print("Durbin-Watson:", dw)

    return dw

#========================================================
# ECUACIÓN DEL MODELO
#========================================================

def f_ecuacion_modelo(modelo,
                      nombres_variables,
                      redondeo = 4):

    intercepto = round(modelo.intercept_, redondeo)

    coefs = np.round(modelo.coef_, redondeo)

    ecuacion = f"ŷ = {intercepto}"

    for coef, variable in zip(coefs,nombres_variables):

        signo = "+" if coef >= 0 else "-"

        ecuacion += f" {signo} {abs(coef)}*{variable}"

    print(ecuacion)

    return ecuacion

#========================================================
# EVALUACIÓN DE MODELOS
#========================================================

def f_evaluacion(modelos,
                 X_validacion_list,
                 y_validacion_list,
                 nombres = None):

    if not isinstance(modelos, list):
        modelos = [modelos]

    resultados = []

    for i, modelo in enumerate(modelos):

        X_val = X_validacion_list[i]

        y_real = y_validacion_list[i]

        pred = modelo.predict(X_val)

        mse = mean_squared_error(y_real,
                                 pred)

        rmse = np.sqrt(mse)

        mae = mean_absolute_error(y_real,
                                  pred)

        r2 = r2_score(y_real,
                      pred)

        n = len(y_real)

        p = X_val.shape[1]

        r2_adj = 1 - (1-r2)*(n-1)/(n-p-1)

        resultados.append({
            "Modelo": nombres[i],
            "R_square": round(r2,4),
            "R_square_ajustado": round(r2_adj,4),
            "MSE": round(mse,4),
            "RMSE": round(rmse,4),
            "MAE": round(mae,4)
        })

    return pd.DataFrame(resultados)


## Cargar datos


In [7]:
url = "https://raw.githubusercontent.com/rpizarrog/Libro-Aprendizaje-Automatico.-Casos-de-Estudio-con-R-y-Python/refs/heads/main/datos/datos_frijol.csv"
datos = f_cargar_datos(url)

## Visualizar datos


In [11]:
f_visualizar_head_tail_reducido(datos)

,temperatura,humedad,precipitacion,nitrogeno,...,horas_sol,tipo_suelo_arenoso,tipo_suelo_franco,calidad_frijol
0,23.7,62.41,115.66,126.84,...,9,False,False,267
1,20.82,47.46,126.49,85.05,...,9,True,False,209
2,25.93,76.44,122.39,82.71,...,7,True,False,241
3,24.96,63.91,106.36,99.0,...,7,False,False,236
4,29.35,52.56,77.93,136.32,...,9,True,False,260
5,25.89,41.49,111.89,80.13,...,6,False,False,222
6,...,...,...,...,...,...,...,...,...
7,25.19,57.35,99.24,141.5,...,8,True,False,272
8,26.8,47.17,128.42,132.78,...,9,False,False,258
9,32.07,53.04,70.02,54.24,...,9,True,False,214
